# B2: Evaluation

**Goal.** Evaluate all methods under the same rules on the 50 shared TriviaQA questions (Qwen2.5-0.5B): AUROC, AURC with randomly broken ties, and a bootstrap interval for the AUROC.

In [3]:
import sys

sys.path.insert(0, "..")  # so that `common` can be imported from notebooks/

import numpy as np
import pandas as pd

from common.data import load_data
from common.evaluate import evaluate
from common.summary import summary_table

rows = load_data("qwen0.5b", 50)
correct = np.array([row["correct"] for row in rows])
print("questions:", len(rows), "| correct:", int(correct.sum()), "| wrong:", int((~correct).sum()))


questions: 50 | correct: 12 | wrong: 38


## Implementation

`common/io.py` checks every score file against `CONTRACT.md`. `common/evaluate.py` computes AUROC and AURC (probly's `selective_prediction`, ties broken randomly over 20 shuffles) and a bootstrap interval. First check: fake scores must behave as expected.


In [4]:
fake = {
    "perfect": (~correct).astype(float),
    "random": np.random.default_rng(1).random(len(correct)),
    "constant": np.zeros(len(correct)),
    "inverted": correct.astype(float),
}
pd.DataFrame({name: evaluate(scores, correct) for name, scores in fake.items()}).T.round(3)

,auroc,aurc
perfect,1.000,0.466
random,0.526,0.758
constant,0.500,0.753
inverted,0.000,0.964
